# LangChain: Q&A over Documents

## Goal of this notebook

Large Language Models are powerful, but they cannot directly access external documents unless we provide them as context.

To solve this problem we use **Retrieval-Augmented Generation (RAG)**.

RAG systems combine:

- document retrieval
- vector search
- language models

In this notebook we will build a Question-Answering system that can answer questions based on a collection of documents.

The workflow will include:

1. Loading documents
2. Splitting documents into chunks
3. Creating embeddings
4. Storing embeddings in a vector database
5. Retrieving relevant documents
6. Generating answers using an LLM


### RAG Architecture:

The pipeline used in this notebook follows this architecture:

User Question  
        ↓  
Question Embedding  
        ↓  
Vector Search  
        ↓  
Relevant Documents  
        ↓  
Prompt + LLM  
        ↓  
Final Answer

In [2]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

## Environment Setup

We start by loading the required libraries and configuring the OpenAI API key.

This notebook uses:

- LangChain
- OpenAI embeddings
- a vector database for semantic search

In [2]:
# Use current model names to avoid migration warnings in modern LangChain.
llm_model = "gpt-4o-mini"
embedding_model = "text-embedding-3-small"

In [3]:
from IPython.display import display, Markdown

from langchain_community.document_loaders import CSVLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

## Loading Documents

In this step we load the source documents that will be used for the question-answering system.

These documents represent the knowledge base that the model will use to answer questions.

However, raw documents are often too large to be processed directly, so we need to split them into smaller pieces.

In [4]:
file = 'OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file)

In [5]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


qa_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question.
If the answer is not in the context, say you do not know.

Context:
{context}

Question: {question}
""")

In [6]:
docs = loader.load()
embeddings = OpenAIEmbeddings(model=embedding_model)
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()

In [7]:
query ="Please list all your shirts with sun protection \
in a table in markdown and summarize each one."

## Splitting Documents into Chunks

Large documents must be divided into smaller chunks before creating embeddings.

This step is important because:

- embeddings work best on smaller text segments
- smaller chunks improve retrieval accuracy
- LLM context windows are limited

LangChain provides text splitters that break documents into manageable chunks.

In [8]:
llm = ChatOpenAI(model=llm_model, temperature=0.0)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | qa_prompt
    | llm
    | StrOutputParser()
)

response = rag_chain.invoke(query)

In [9]:
display(Markdown(response))

Here is a table summarizing the shirts with sun protection:

| Name                                   | Description                                                                                                           |
|----------------------------------------|-----------------------------------------------------------------------------------------------------------------------|
| Sun Shield Shirt                       | High-performance sun shirt with UPF 50+ protection, made of 78% nylon and 22% Lycra Xtra Life. Moisture-wicking, abrasion-resistant, and fits over swimsuits. Recommended by The Skin Cancer Foundation. |
| Men's Tropical Plaid Short-Sleeve Shirt| Lightest hot-weather shirt with UPF 50+ protection, made of 100% polyester. Features a relaxed fit, cape venting, and two front bellows pockets. Wrinkle-resistant. |
| Men's Plaid Tropic Shirt, Short-Sleeve | Ultracomfortable shirt with UPF 50+ protection, made of 52% polyester and 48% nylon. Designed for fishing, features cape venting, two front bellows pockets, and is machine washable. |
| Tropical Breeze Shirt                  | Lightweight, breathable long-sleeve shirt with UPF 50+ protection, made of 71% nylon and 29% polyester. Features moisture-wicking, wrinkle-resistant fabric, cape venting, and is machine washable. |

Each shirt offers high sun protection (UPF 50+) and is designed for comfort and functionality in hot weather.

## Creating Embeddings

Embeddings convert text into numerical vectors that capture semantic meaning.

Texts with similar meanings will have embeddings that are close to each other in vector space.

These vectors allow us to perform **semantic search** instead of simple keyword matching.

In [10]:
loader = CSVLoader(file_path=file)

In [11]:
docs = loader.load()

In [12]:
docs[0]

Document(metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 0}, page_content=": 0\nname: Women's Campside Oxfords\ndescription: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. \n\nSize & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. \n\nSpecs: Approx. weight: 1 lb.1 oz. per pair. \n\nConstruction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. \n\nQuestions? Please contact us for any inquiries.")

In [13]:
embeddings = OpenAIEmbeddings(model=embedding_model)

In [14]:
embed = embeddings.embed_query("Hi my name is Harrison")

## Storing Embeddings in a Vector Database

After generating embeddings, we store them in a vector database.

A vector database enables fast similarity search between vectors.

When a user asks a question, we will convert the question into an embedding and retrieve the most relevant document chunks.

In [15]:
print(len(embed))

1536


In [16]:
print(embed[:5])

[0.0056793042458593845, -0.005919360555708408, -0.06266986578702927, 0.022632135078310966, -0.05134164169430733]


In [17]:
db = FAISS.from_documents(
    docs,
    embeddings,
)

In [18]:
query = "Please suggest a shirt with sunblocking"

## Retrieving Relevant Documents

When a user asks a question, the system performs the following steps:

1. Convert the question into an embedding
2. Search the vector database
3. Retrieve the most relevant document chunks

These retrieved chunks will then be passed to the language model as context.

In [19]:
docs = db.similarity_search(query)

In [20]:
len(docs)

4

In [21]:
docs[0]

Document(id='c3f6a050-c741-46a9-97d1-0fc5db568536', metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 255}, page_content=': 255\nname: Sun Shield Shirt by\ndescription: "Block the sun, not the fun – our high-performance sun shirt is guaranteed to protect from harmful UV rays. \n\nSize & Fit: Slightly Fitted: Softly shapes the body. Falls at hip.\n\nFabric & Care: 78% nylon, 22% Lycra Xtra Life fiber. UPF 50+ rated – the highest rated sun protection possible. Handwash, line dry.\n\nAdditional Features: Wicks moisture for quick-drying comfort. Fits comfortably over your favorite swimsuit. Abrasion resistant for season after season of wear. Imported.\n\nSun Protection That Won\'t Wear Off\nOur high-performance fabric provides SPF 50+ sun protection, blocking 98% of the sun\'s harmful rays. This fabric is recommended by The Skin Cancer Foundation as an effective UV protectant.')

In [22]:
retriever = db.as_retriever()

In [23]:
llm = ChatOpenAI(model=llm_model, temperature=0.0)

In [24]:
qdocs = format_docs(docs)


## Building a Question Answering Chain

The retrieved documents alone are not enough.

We still need a language model to:

- read the retrieved context
- understand the question
- generate a coherent answer

LangChain provides QA chains that combine retrieved documents with the question and pass them to the LLM.

In [25]:
manual_qa_prompt = ChatPromptTemplate.from_template("""Use the following product information to answer the question.
Return the result in markdown when appropriate.

Context:
{context}

Question: {question}
""")

manual_qa_chain = manual_qa_prompt | llm | StrOutputParser()

response = manual_qa_chain.invoke(
    {
        "context": qdocs,
        "question": "Please list all your shirts with sun protection in a table in markdown and summarize each one.",
    }
)


In [26]:
display(Markdown(response))

Here’s a summary of the shirts with sun protection in a markdown table format:

| Name                                   | Description                                                                                                           | Fabric Composition                | UPF Rating | Additional Features                                                                                     |
|----------------------------------------|-----------------------------------------------------------------------------------------------------------------------|-----------------------------------|------------|---------------------------------------------------------------------------------------------------------|
| Sun Shield Shirt                       | High-performance sun shirt that protects from harmful UV rays. Softly shapes the body and falls at the hip.          | 78% nylon, 22% Lycra Xtra Life    | UPF 50+    | Wicks moisture, abrasion resistant, fits over swimsuits, recommended by The Skin Cancer Foundation.     |
| Men's Tropical Plaid Short-Sleeve Shirt| Lightest hot-weather shirt with a relaxed fit. Features front and back cape venting for cool breezes.                | 100% polyester                     | UPF 50+    | Wrinkle-resistant, two front bellows pockets, provides superior sun protection.                         |
| Men's Plaid Tropic Shirt, Short-Sleeve | Ultracomfortable shirt originally designed for fishing. Offers UPF 50+ coverage and evaporates perspiration quickly. | 52% polyester, 48% nylon          | UPF 50+    | Wrinkle-free, front and back cape venting, two front bellows pockets, great for extended travel.       |
| Tropical Breeze Shirt                  | Lightweight, breathable long-sleeve shirt with superior SunSmart protection. Keeps you cool and comfortable.         | 71% nylon, 29% polyester          | UPF 50+    | Wrinkle-resistant, moisture-wicking, front and back cape venting, designed for outdoor activities.      |

This table summarizes the key features and specifications of each shirt, highlighting their sun protection capabilities.

## Running the Question-Answering System

Finally, we run the QA pipeline.

The system:

1. retrieves relevant documents
2. inserts them into the prompt
3. generates an answer using the LLM

This approach significantly improves answer accuracy compared to using the model alone.

In [27]:
qa_stuff = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | qa_prompt
    | llm
    | StrOutputParser()
)

In [28]:
query =  "Please list all your shirts with sun protection in a table \
in markdown and summarize each one."

In [29]:
response = qa_stuff.invoke(query)

In [30]:
display(Markdown(response))

Here is a table summarizing the shirts with sun protection:

| Name                                   | Description                                                                                                           |
|----------------------------------------|-----------------------------------------------------------------------------------------------------------------------|
| Sun Shield Shirt                       | High-performance sun shirt with UPF 50+ protection, blocking 98% of UV rays. Made of 78% nylon and 22% Lycra. Moisture-wicking, abrasion-resistant, and fits over swimsuits. Handwash recommended. |
| Men's Tropical Plaid Short-Sleeve Shirt| Lightest hot-weather shirt with UPF 50+ protection. Made of 100% polyester, wrinkle-resistant, with cape venting and two front pockets. Blocks 98% of UV rays. |
| Men's Plaid Tropic Shirt, Short-Sleeve | Ultracomfortable shirt rated UPF 50+, designed for fishing. Made of 52% polyester and 48% nylon, wrinkle-free, and moisture-evaporating. Features cape venting and two front pockets. |
| Tropical Breeze Shirt                  | Lightweight, breathable long-sleeve shirt with UPF 50+ protection. Made of 71% nylon and 29% polyester. Wrinkle-resistant, moisture-wicking, and designed for outdoor activities. Features cape venting and two front pockets. |

Each shirt offers high sun protection with UPF 50+ ratings, blocking 98% of harmful UV rays, and includes features for comfort and convenience.

In [31]:
response = rag_chain.invoke(query)

In [32]:
vectorstore = FAISS.from_documents(
    loader.load(),
    embeddings,
)
retriever = vectorstore.as_retriever()

## Conclusion

In this notebook we built a basic Retrieval-Augmented Generation (RAG) system using LangChain.

Key takeaways:

- LLMs cannot access external knowledge unless it is provided in the prompt.
- Embeddings allow us to represent text as vectors capturing semantic meaning.
- Vector databases enable efficient similarity search.
- Retrieval-Augmented Generation combines document retrieval with LLM reasoning.

This architecture is widely used in real-world AI systems such as document assistants, knowledge base chatbots, and enterprise search tools.